In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import tomotopy as tp
from tomotopy.utils import Corpus
from tomotopy.coherence import Coherence
import matplotlib.pyplot as plt
import seaborn as sns
import pyLDAvis

In [3]:
df = pd.read_csv("cleaned_corpus.csv", index_col = 0, parse_dates = ['approved_date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 994 entries, repacts:datasets/markdown/repacts/2016/ra_10923_2016.md to repacts:datasets/markdown/repacts/2022/ra_11933_2022.md
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   source           994 non-null    object        
 1   year             994 non-null    int64         
 2   month            994 non-null    object        
 3   path             994 non-null    object        
 4   basename         994 non-null    object        
 5   title            994 non-null    object        
 6   day              994 non-null    int64         
 7   year_extracted   994 non-null    int64         
 8   approved_date    994 non-null    datetime64[ns]
 9   cleaned_content  994 non-null    object        
dtypes: datetime64[ns](1), int64(3), object(6)
memory usage: 85.4+ KB


In [4]:
df['quarter'] = df.approved_date.dt.quarter
# q1 = df[df.quarters == 1]
# q2 = df[df.quarters == 2]
# q3 = df[df.quarters == 3]
# q4 = df[df.quarters == 4]

In [33]:
# TODO: [Visual] bar graph of how many documents per quarter

years = set(df['year'])
quarters = set(df['quarter'])
models = {}
for y in years:
    for q in quarters:
        # collect into corpus obj
        corpus = Corpus()
        period = df[(df.year == y) & (df.quarter == q)] # pyright: ignore[reportUnknownVariableType]
        print(f"found {len(period)} records for year {y} quarter {q}")
        if len(period):
            for text in period['cleaned_content']: # pyright: ignore[reportUnknownVariableType]
                corpus.add_doc(text.split()) # pyright: ignore[reportUnknownArgumentType, reportUnknownMemberType]

            # training the model
            seed = 357
            # k = 5 topics
            model = tp.LDAModel(k = 5, corpus = corpus, seed = seed)
            iters = 1000
            model.train(iterations=iters, workers=1) # pyright: ignore[reportCallIssue, reportUnknownMemberType]
            models[(y, q)] = model

            sample_docs = 5
            if len(period) > sample_docs:
                idx = 0
                list_period = period.index.tolist()
                for doc in model.docs:
                    if idx > sample_docs:
                        break
                    print(f"idx: {idx}")
                    print(f"in period: {list_period[idx]}")
                    top_topics = doc.get_topics(top_n = 1)
                    topic, score = top_topics[0]
                    print(f"{df.loc[list_period[idx], 'title']}: #{topic} ({score:0.2f}%)")
                    idx += 1
        else:
            print("skipping, no records")

len(models)

found 0 records for year 2016 quarter 1
skipping, no records
found 0 records for year 2016 quarter 2
skipping, no records
found 0 records for year 2016 quarter 3
skipping, no records
found 1 records for year 2016 quarter 4
found 0 records for year 2017 quarter 1
skipping, no records
found 2 records for year 2017 quarter 2
found 24 records for year 2017 quarter 3
idx: 0
in period: repacts:datasets/markdown/repacts/2017/ra_10927_2017.md
Ra 10927 2017: #0 (0.71%)
idx: 1
in period: repacts:datasets/markdown/repacts/2017/ra_10928_2017.md
Ra 10928 2017: #0 (0.66%)
idx: 2
in period: repacts:datasets/markdown/repacts/2017/ra_10930_2017.md
Ra 10930 2017: #0 (0.63%)
idx: 3
in period: repacts:datasets/markdown/repacts/2017/ra_10931_2017.md
Ra 10931 2017: #3 (0.75%)
idx: 4
in period: repacts:datasets/markdown/repacts/2017/ra_10932_2017.md
Ra 10932 2017: #3 (0.48%)
idx: 5
in period: repacts:datasets/markdown/repacts/2017/ra_10933_2017.md
Ra 10933 2017: #1 (0.82%)
found 14 records for year 2017 quar

23

In [35]:
def top_words(model, k):
    """Print the top words for topic k in a model."""
    top_words = model.get_topic_words(topic_id = k, top_n = 5)
    top_words = [f"{word} ({score:0.4f}%)" for (word, score) in top_words]
    print(f"Topic {k}: {', '.join(top_words)}")

for k,model in models.items(): # pyright: ignore[reportUnknownVariableType]
    year, quarter = k # pyright: ignore[reportUnknownVariableType]
    print(f"topics for year {year} quarter {quarter}")
    for i in range(model.k): # pyright: ignore[reportUnknownArgumentType, reportUnknownMemberType]
        top_words(model, i) # pyright: ignore[reportUnknownArgumentType]

# TODO: consult excluding numbers, days of the week/month (incl. 'day'), other senate terms (article, senate, representative [excluding house kasi it can mean 2 things])
#ALS naging al https://lawphil.net/statutes/repacts/ra2020/ra_11510_2020.html

topics for year 2016 quarter 4
Topic 0: hold (0.0899%), general (0.0720%), provide (0.0540%), sec (0.0540%), office (0.0540%)
Topic 1: sangguniang (0.2247%), thousand (0.0662%), provision (0.0662%), rule (0.0530%), appropriate (0.0398%)
Topic 2: election (0.1781%), kabataan (0.1594%), barangay (0.1407%), follow (0.0469%), peso (0.0376%)
Topic 3: hundred (0.1166%), million (0.0584%), house (0.0584%), elect (0.0584%), immediately (0.0438%)
Topic 4: appropriation (0.1050%), new (0.0631%), know (0.0631%), qualify (0.0421%), appointment (0.0421%)
topics for year 2017 quarter 2
Topic 0: sec (0.0774%), public (0.0608%), station (0.0553%), provide (0.0442%), house (0.0415%)
Topic 1: franchise (0.1306%), service (0.0523%), provision (0.0314%), effectivity (0.0314%), use (0.0314%)
Topic 2: system (0.0338%), inc (0.0307%), communication (0.0276%), person (0.0276%), labor (0.0215%)
Topic 3: telecommunication (0.0568%), read (0.0426%), follow (0.0426%), tax (0.0355%), operate (0.0320%)
Topic 4: gra

comparing the topic words versus the top topic for some sample docs, I think I need to spread the topics more because for https://lawphil.net/statutes/repacts/ra2017/ra_10927_2017.html the focus was "An Act Designating Casinos as Covered Persons Under Republic Act No. 9160, Otherwise Known as the "Anti-Money Laundering Act of 2001", as Amended" but the top words were "Topic 0: case (0.0260%), provision (0.0180%), provide (0.0172%), follow (0.0144%), law (0.0144%)", or use them as stopwords (check TF-IDF first pano magwowork)